# 04 — Training & Experiments

**Lab 2 — Group 4 | CIFAR-10**

Bước 6: huấn luyện, điều chỉnh hyperparameters, so sánh VGG16 và DenseNet121. Giữ ma trận E1–E5 gốc; bổ sung ghi log theo batch và epoch để notebook 05 theo dõi. Mỗi notebook chạy kernel riêng; cấu hình trong cell code đầu tiên.

In [1]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs').is_dir() and (p/'src').is_dir()), None)
assert ROOT is not None, 'Hãy mở notebook trong thư mục dự án đã giải nén.'
sys.path.insert(0, str(ROOT))
# Cấu hình trực tiếp trong notebook (không cần file cấu hình riêng).
MODE = "review"  # review: kết quả gốc; full: chạy thực tế
RUN_ID = "run_01"  # notebook 04–06 phải dùng cùng tên khi chạy full
DEVICE_OVERRIDE = "auto"  # auto / cpu / cuda
CPU_THREADS = 1
EPOCHS_OVERRIDE = None  # None: epoch gốc; 5: khảo sát nhiều epoch
LIVE = MODE == "full"
assert MODE in ("review", "full")
DOWNLOAD = True
import io, json, random, hashlib, time, gc
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

DATA_DIR = ROOT / 'data'
SESSION_DIR = ROOT / 'outputs' / 'notebook_runs' / RUN_ID
FIG_DIR = SESSION_DIR / 'figures'
SAVE_FIGS = LIVE

def save_fig(name):
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")

random.seed(SEED)
np.random.seed(SEED)
if LIVE:
    import torch
    import torchvision
    from torch import nn
    from torch.utils.data import DataLoader, Subset, random_split
    from torchvision import transforms, models
    from PIL import Image
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from torch.utils.tensorboard import SummaryWriter
    import torch.nn.functional as F

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    set_seed(SEED)
    DEVICE = torch.device(("cuda" if torch.cuda.is_available() else "cpu")
                          if DEVICE_OVERRIDE == "auto" else DEVICE_OVERRIDE)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA không khả dụng. Chọn CPU hoặc sửa môi trường.")
    if DEVICE.type == "cpu":
        torch.set_num_threads(CPU_THREADS)
    print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
    print("Device:", DEVICE)
    print("Data:", DATA_DIR, "| Output mới:", SESSION_DIR)
else:
    print("Review: không cần import PyTorch, không tạo file training.")

Review: không cần import PyTorch, không tạo file training.


## 6.1. Training loop và ghi log monitoring
CrossEntropyLoss; optimizer theo cấu hình; forward → loss → backward → optimizer step. Validation không cập nhật trọng số. Chọn checkpoint theo validation accuracy cao nhất, rồi validation loss thấp nhất. Code bên dưới dựa trên `src/train.py` của nhóm, bổ sung ghi log; không sửa module gốc. History được lưu sau mỗi epoch. TensorBoard có thể mở ở một terminal khác ngay khi chạy training.

In [2]:
if LIVE:
    """TV3 - Training and validation pipeline for CIFAR-10."""

    import math
    import time
    from pathlib import Path
    from typing import Any

    import torch
    from torch import nn
    from torch.optim import Adam, AdamW, SGD
    from torch.utils.tensorboard import SummaryWriter


    def _plain_config(value: Any) -> Any:
        """Convert config values to basic serializable Python types."""
        if isinstance(value, dict):
            return {
                str(key): _plain_config(item)
                for key, item in value.items()
            }

        if isinstance(value, (list, tuple)):
            return [_plain_config(item) for item in value]

        if isinstance(value, (Path, torch.device)):
            return str(value)

        if value is None or isinstance(value, (str, int, float, bool)):
            return value

        raise TypeError(
            f"Unsupported config value: {type(value).__name__}"
        )


    def _positive_int(config: dict, key: str) -> int:
        """Read and validate a positive integer."""
        value = config.get(key)

        if (
            isinstance(value, bool)
            or not isinstance(value, int)
            or value <= 0
        ):
            raise ValueError(
                f"config['{key}'] must be a positive integer; "
                f"received {value!r}"
            )

        return value


    def _make_optimizer(
        model: nn.Module,
        config: dict,
        learning_rate: float,
        weight_decay: float,
    ):
        """Create an optimizer using trainable parameters only."""
        parameters = [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ]

        if not parameters:
            raise ValueError(
                "No trainable parameters found. "
                "Check the model freeze/fine-tuning configuration."
            )

        name = str(config.get("optimizer") or "adam").lower()

        if name == "adam":
            return Adam(
                parameters,
                lr=learning_rate,
                weight_decay=weight_decay,
            )

        if name == "adamw":
            return AdamW(
                parameters,
                lr=learning_rate,
                weight_decay=weight_decay,
            )

        if name == "sgd":
            momentum = float(config.get("momentum", 0.9))

            if not math.isfinite(momentum) or not 0 <= momentum < 1:
                raise ValueError("momentum must be in [0, 1).")

            return SGD(
                parameters,
                lr=learning_rate,
                momentum=momentum,
                weight_decay=weight_decay,
            )

        raise ValueError(
            f"Unsupported optimizer {name!r}. "
            "Use adam, adamw, or sgd."
        )


    def _keep_frozen_batchnorm_eval(model: nn.Module) -> None:
        """Prevent frozen BatchNorm layers from updating running statistics."""
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                parameters = list(module.parameters(recurse=False))

                if all(not parameter.requires_grad for parameter in parameters):
                    module.eval()


    def _run_epoch(
        model: nn.Module,
        loader,
        loss_fn: nn.Module,
        device: torch.device,
        optimizer=None,
        writer=None,
        epoch=1,
    ) -> dict[str, float]:
        """Run one training or validation epoch."""
        training = optimizer is not None
        model.train(training)

        if training:
            _keep_frozen_batchnorm_eval(model)

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        phase = "Train" if training else "Validation"
        print(f"{phase}: bắt đầu {len(loader)} batch", flush=True)

        for batch_index, (images, labels) in enumerate(loader, start=1):
            images = images.to(device)
            labels = labels.to(device=device, dtype=torch.long)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(training):
                logits = model(images)

                if (
                    logits.ndim != 2
                    or logits.shape[0] != labels.size(0)
                    or logits.shape[1] != 10
                ):
                    raise ValueError(
                        "Expected model logits with shape [B, 10], "
                        f"received {tuple(logits.shape)}."
                    )

                loss = loss_fn(logits, labels)

                if not torch.isfinite(loss).item():
                    raise ValueError("Non-finite loss encountered.")

                if training:
                    loss.backward()
                    optimizer.step()

            count = labels.size(0)

            # Weight losses by sample count, including the final smaller batch.
            total_loss += float(loss.detach().item()) * count
            total_correct += int(
                (logits.detach().argmax(dim=1) == labels).sum().item()
            )
            total_samples += count

            if batch_index == 1 or batch_index % 50 == 0 or batch_index == len(loader):
                if writer is not None:
                    step = (epoch - 1) * len(loader) + batch_index
                    split = "train" if training else "val"
                    writer.add_scalar(f"Batch/{split}_loss", float(loss.detach().item()), step)
                    writer.add_scalar(f"Batch/{split}_running_accuracy", total_correct / total_samples, step)
                    writer.flush()
                print(
                    f"{phase}: batch {batch_index}/{len(loader)} | "
                    f"loss={total_loss / total_samples:.4f} | "
                    f"accuracy={total_correct / total_samples:.4f}",
                    flush=True,
                )

        if total_samples == 0:
            raise ValueError("The DataLoader is empty.")

        return {
            "loss": float(total_loss / total_samples),
            "accuracy": float(total_correct / total_samples),
        }


    def train_model(
        model,
        dataloaders: dict[str, Any],
        config: dict[str, Any],
    ) -> dict[str, Any]:
        """
        Train and validate a CIFAR-10 model.

        Required DataLoader keys: train and val.

        Returns:
            history
            best_checkpoint_path
            best_epoch
            best_val_accuracy
        """
        if not isinstance(config, dict):
            raise TypeError("config must be a dictionary.")

        for split in ("train", "val"):
            if split not in dataloaders or dataloaders[split] is None:
                raise KeyError(
                    f"dataloaders must contain a '{split}' DataLoader."
                )

        epochs = _positive_int(config, "epochs")
        batch_size = _positive_int(config, "batch_size")

        if config.get("learning_rate") is None:
            raise ValueError("config['learning_rate'] cannot be null.")

        learning_rate = float(config["learning_rate"])
        weight_decay = float(config.get("weight_decay") or 0.0)

        if not math.isfinite(learning_rate) or learning_rate <= 0:
            raise ValueError("learning_rate must be positive and finite.")

        if not math.isfinite(weight_decay) or weight_decay < 0:
            raise ValueError(
                "weight_decay must be finite and non-negative."
            )

        device_name = str(
            config.get("device")
            or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        if device_name == "auto":
            device_name = (
                "cuda" if torch.cuda.is_available() else "cpu"
            )

        device = torch.device(device_name)

        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA was requested but is unavailable."
            )

        experiment_name = str(
            config.get("experiment_name") or "training_run"
        )

        run_dir = Path(
            config.get("run_dir")
            or Path("outputs") / "runs" / experiment_name
        )
        tensorboard_dir = Path(
            config.get("tensorboard_dir")
            or run_dir / "tensorboard"
        )
        checkpoint_dir = Path(
            config.get("checkpoint_dir")
            or run_dir / "checkpoints"
        )

        run_dir.mkdir(parents=True, exist_ok=True)
        tensorboard_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        checkpoint_path = checkpoint_dir / "best_model.pth"

        model = model.to(device)
        loss_fn = nn.CrossEntropyLoss()

        optimizer = _make_optimizer(
            model=model,
            config=config,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )

        checkpoint_config = _plain_config(dict(config))
        checkpoint_config["device"] = str(device)
        checkpoint_config["batch_size"] = batch_size

        history = []
        best_epoch = None
        best_val_accuracy = float("-inf")
        best_val_loss = float("inf")
        start_time = time.perf_counter()

        writer = SummaryWriter(log_dir=str(tensorboard_dir))

        try:
            for epoch in range(1, epochs + 1):
                epoch_start = time.perf_counter()
                if device.type == "cuda":
                    torch.cuda.reset_peak_memory_stats(device)
                train_metrics = _run_epoch(
                    model=model,
                    loader=dataloaders["train"],
                    loss_fn=loss_fn,
                    device=device,
                    optimizer=optimizer, writer=writer, epoch=epoch,
                )

                val_metrics = _run_epoch(
                    model=model,
                    loader=dataloaders["val"],
                    loss_fn=loss_fn,
                    device=device, writer=writer, epoch=epoch,
                )

                row = {
                    "epoch": epoch,
                    "train_loss": float(train_metrics["loss"]),
                    "train_accuracy": float(train_metrics["accuracy"]),
                    "val_loss": float(val_metrics["loss"]),
                    "val_accuracy": float(val_metrics["accuracy"]),
                }

                if not all(
                    math.isfinite(row[key])
                    for key in (
                        "train_loss",
                        "train_accuracy",
                        "val_loss",
                        "val_accuracy",
                    )
                ):
                    raise ValueError(
                        f"Non-finite metric at epoch {epoch}."
                    )

                row["learning_rate"] = float(optimizer.param_groups[0]["lr"])
                row["epoch_time_sec"] = time.perf_counter() - epoch_start
                row["accuracy_gap"] = row["train_accuracy"] - row["val_accuracy"]
                row["loss_gap"] = row["val_loss"] - row["train_loss"]
                if device.type == "cuda":
                    row["gpu_peak_memory_mb"] = torch.cuda.max_memory_allocated(device) / 1024**2
                history.append(row)
                pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
                for key in ("learning_rate", "epoch_time_sec", "accuracy_gap", "loss_gap", "gpu_peak_memory_mb"):
                    if key in row:
                        writer.add_scalar(f"Monitoring/{key}", row[key], epoch)

                writer.add_scalar("Loss/train", row["train_loss"], epoch)
                writer.add_scalar("Loss/val", row["val_loss"], epoch)
                writer.add_scalar(
                    "Accuracy/train", row["train_accuracy"], epoch
                )
                writer.add_scalar(
                    "Accuracy/val", row["val_accuracy"], epoch
                )
                writer.flush()

                print(
                    f"Epoch [{epoch}/{epochs}] "
                    f"train_loss={row['train_loss']:.4f} "
                    f"train_acc={row['train_accuracy']:.4f} "
                    f"val_loss={row['val_loss']:.4f} "
                    f"val_acc={row['val_accuracy']:.4f}"
                )

                # Select by highest validation accuracy, then lowest val loss.
                is_better = (
                    best_epoch is None
                    or row["val_accuracy"] > best_val_accuracy
                    or (
                        row["val_accuracy"] == best_val_accuracy
                        and row["val_loss"] < best_val_loss
                    )
                )

                if is_better:
                    best_epoch = epoch
                    best_val_accuracy = row["val_accuracy"]
                    best_val_loss = row["val_loss"]

                    torch.save(
                        {
                            "model_state_dict": model.state_dict(),
                            "epoch": int(epoch),
                            "val_accuracy": float(best_val_accuracy),
                            "val_loss": float(best_val_loss),
                            "config": checkpoint_config,
                        },
                        checkpoint_path,
                    )

        finally:
            writer.close()

        if best_epoch is None or not checkpoint_path.is_file():
            raise RuntimeError(
                "Training finished without saving a checkpoint."
            )

        elapsed = time.perf_counter() - start_time

        print(f"Best epoch: {best_epoch}")
        print(f"Best validation accuracy: {best_val_accuracy:.4f}")
        print(f"Checkpoint saved to: {checkpoint_path}")
        print(f"Training time: {elapsed:.2f} seconds")

        return {
            "history": history,
            "best_checkpoint_path": str(checkpoint_path),
            "best_epoch": int(best_epoch),
            "best_val_accuracy": float(best_val_accuracy),
        }

## 6.2. Ma trận thí nghiệm
E1 vs E2: kiến trúc; E2 vs E3: learning rate; E2 vs E4: batch size; E3 vs E5: fine-tuning. E5 10 epoch chưa có kết quả, không đưa vào 5 run tái lập.

In [3]:
import yaml
RUN_IDS = ['E1_vgg16','E2_densenet121','E3_learning_rate','E4_batch_size','E5_cpu_1epoch']
configs = [yaml.safe_load((ROOT/'configs'/f'{name}.yaml').read_text(encoding='utf-8-sig')) for name in RUN_IDS]
display(pd.DataFrame(configs)[['experiment_name','model_name','strategy','learning_rate','batch_size','epochs','seed']])

,experiment_name,model_name,strategy,learning_rate,batch_size,epochs,seed
0,E1_vgg16,vgg16,classifier_only,0.0010,32,1,42
1,E2_densenet121,densenet121,classifier_only,0.0010,32,1,42
2,E3_learning_rate,densenet121,classifier_only,0.0001,32,1,42
3,E4_batch_size,densenet121,classifier_only,0.0010,64,1,42
4,E5_cpu_1epoch,densenet121,finetune_last_layers,0.0001,32,1,42


## 6.3. Huấn luyện E1–E5
Chế độ `review` đọc kết quả nhóm đã lưu. `full` huấn luyện thực tế, tạo thư mục mới theo RUN_ID. Có thể đặt `EPOCHS_OVERRIDE = 5` trong cell code đầu tiên của notebook này để khảo sát xu hướng nhiều epoch, mặc định `None` giữ cấu hình gốc (1 epoch). Mở TensorBoard bằng lệnh được in ở cell tiếp theo trước khi bắt đầu train.

In [4]:
print(f'tensorboard --logdir "{SESSION_DIR / "training"}" --port 6006')
print('Mở http://localhost:6006 khi training đã bắt đầu ghi event files.')

tensorboard --logdir "d:\Lab2_DL_Group4-main\outputs\notebook_runs\run_01\training" --port 6006
Mở http://localhost:6006 khi training đã bắt đầu ghi event files.


In [5]:
if LIVE:
    from src.data import build_dataloaders
    from src.models import build_model
    from src.utils import set_seed, validate_config, save_json
    train_root = SESSION_DIR/'training'
    train_root.mkdir(parents=True,exist_ok=False)
    results=[]
    for original in configs:
        cfg=dict(original)
        if EPOCHS_OVERRIDE is not None:
            cfg['epochs'] = EPOCHS_OVERRIDE
        name=cfg['experiment_name']
        folder=train_root/name
        folder.mkdir()
        cfg.update(device=str(DEVICE),data_dir=str(DATA_DIR),run_dir=str(folder),log_dir=str(folder),tensorboard_dir=str(folder/'tensorboard'),checkpoint_dir=str(folder/'checkpoints'))
        validate_config(cfg)
        set_seed(cfg['seed'])
        save_json(folder/'config.json',cfg)
        loaders=build_dataloaders(cfg)
        model=build_model(cfg)
        started=time.perf_counter()
        result=train_model(model,loaders,cfg)
        history=pd.DataFrame(result['history'])
        history.to_csv(folder/'history.csv',index=False)
        best_row=history.loc[history.epoch==result['best_epoch']].iloc[0]
        metrics=dict(experiment_name=name,model_name=cfg['model_name'],strategy=cfg['strategy'],learning_rate=cfg['learning_rate'],batch_size=cfg['batch_size'],epochs=cfg['epochs'],seed=cfg['seed'],best_val_loss=float(best_row.val_loss),best_val_accuracy=float(best_row.val_accuracy),training_time_sec=time.perf_counter()-started,checkpoint_path=str(Path(result['best_checkpoint_path']).relative_to(ROOT)))
        save_json(folder/'metrics.json',metrics)
        results.append(metrics)
        del model,loaders,result
        gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    summary=pd.DataFrame(results)
    summary.to_csv(train_root/'experiment_summary.csv',index=False)
    print('Nguồn: thực nghiệm mới',train_root)
else:
    summary=pd.read_csv(ROOT/'outputs/experiment_summary.csv')
    print('Nguồn: kết quả đo đã lưu trong ZIP gốc. E5 10 epoch chưa thực nghiệm.')
display(summary)

Nguồn: kết quả đo đã lưu trong ZIP gốc. E5 10 epoch chưa thực nghiệm.


,experiment_name,model_name,strategy,learning_rate,batch_size,epochs,seed,best_val_loss,best_val_accuracy,test_accuracy,training_time_sec
0,E1_vgg16,vgg16,classifier_only,0.0010,32,1,42,0.524489,0.8288,NaN,21929.006920
1,E2_densenet121,densenet121,classifier_only,0.0010,32,1,42,0.346216,0.8784,NaN,5407.230531
2,E3_learning_rate,densenet121,classifier_only,0.0001,32,1,42,0.577797,0.8344,NaN,5329.618331
3,E4_batch_size,densenet121,classifier_only,0.0010,64,1,42,0.367412,0.8716,NaN,5499.967411
4,E5_cpu_1epoch,densenet121,finetune_last_layers,0.0001,32,1,42,0.232135,0.9210,0.9181,6135.234429
5,E5_finetuning,densenet121,finetune_last_layers,0.0001,32,10,42,NaN,NaN,NaN,NaN


**Tiếp theo:** notebook 05 đọc history và TensorBoard để theo dõi mô hình; notebook 06 chọn checkpoint và đánh giá test.